# Stephenson + COMBAT integration: patient-level representations across methods\n\nThis notebook builds sample-level representations of the integrated **Stephenson + COMBAT** COVID-19 dataset (`patpy.datasets.combat_stephenson`) using a panel of complementary methods, then collects every embedding into a single `meta_adata` whose `.X` is the **baseline mean pseudobulk** of log-normalised expression and whose `.obsm` carries the alternatives.\n\n**Representations gathered**\n\n| `meta_adata` slot | method | notes |\n|---|---|---|\n| `.X` / `obsm[\"X_pseudobulk_lognorm\"]` | mean pseudobulk on log-normalised counts | baseline |\n| `obsm[\"X_pseudobulk_scanvi\"]`         | mean pseudobulk on the dataset-shipped scANVI embedding (`X_scANVI_dataset`) | latent-space baseline |\n| `obsm[\"X_composition_clr\"]`           | `patpy.tl.CellGroupComposition(apply_clr=True)` | SETA-style cell-type composition |\n| `obsm[\"X_gloscope_py\"]`               | `patpy.tl.GloScope_py(use_gpu=…)` MDS embedding | GPU when cuML is available |\n| `obsm[\"X_sampleclr_ssl\"]`             | SampleCLR self-supervised (pretrain) | from `lueckenlab/SampleCLR` |\n| `obsm[\"X_sampleclr_ft\"]`              | SampleCLR fine-tuned on `severity_idx` | same model, after `fine_tune` |\n| `obsm[\"X_pascient\"]`                  | `patpy.tl.PaSCient` patient embedding | trained from scratch |\n\n**Reproducibility model.** Every heavy stage is implemented as a function in `scripts/stephenson_combat/pipeline.py` and as a SLURM-runnable script (`scripts/stephenson_combat/0[1-5]_*.py`). The notebook calls those same functions, but for each stage it first checks whether a cached `obsm_<stage>.npz` file is present; if so it loads, otherwise it computes inline. Long jobs (SampleCLR, PaSCient, GloScope GPU) are usually pre-computed via SLURM with the matching `.sbatch` files in that directory.\n\n**Environment.** Conda env `patpy-stephenson` (see `scripts/stephenson_combat/README.md` for the install commands), `patpy` editable install on branch `stephenson-combat-integration`, `sampleclr` installed from `git+https://github.com/lueckenlab/SampleCLR.git@main`."

In [ ]:
# If the env is not yet set up, the cell below installs the two pieces that\n# don't ship in PyPI patpy. Run once, then comment out.\n\n# %pip install -U git+https://github.com/lueckenlab/SampleCLR.git@main\n# %pip install -e /ictstr01/groups/luckylab/workspace/vladimir.shitov/patpy\n# %pip install git+https://github.com/genentech/pascient.git@main\n# %pip install \"cupy-cuda12x\" \"cuml-cu12\"  # only needed for GPU GloScope_py

In [ ]:
import sys\nfrom pathlib import Path\n\nimport anndata as ad\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport scanpy as sc\n\nimport patpy\nimport sampleclr\n\n# pull in the shared pipeline helpers\nREPO_DIR = Path(\"/ictstr01/groups/luckylab/workspace/vladimir.shitov/patpy\")\nsys.path.insert(0, str(REPO_DIR / \"scripts\" / \"stephenson_combat\"))\nimport pipeline as p  # noqa: E402\n\nprint(\"patpy     :\", patpy.__version__)\nprint(\"sampleclr :\", sampleclr.__version__)\nprint(\"scanpy    :\", sc.__version__)\nprint(\"anndata   :\", ad.__version__)\n\nnp.random.seed(p.SEED)

## 1. Load the integrated COVID-19 dataset\n\n`patpy.datasets.combat_stephenson` returns the harmonised Stephenson + COMBAT cohort plus a `DatasetInfo` that names the donor (`sample_id`) and cell-type (`cell_coarse_aligned`) columns. The released file already ships:\n\n- raw counts in `adata.layers[\"X_raw_counts\"]`,\n- a scANVI integration of the two studies in `adata.obsm[\"X_scANVI_dataset\"]`,\n- a UMAP of the same in `adata.obsm[\"X_umap_scANVI_dataset\"]`,\n- per-sample metadata (severity, outcome, site, etc.) already in `adata.obs`.\n\nWe set the default `X_umap` to the scANVI UMAP so `sc.pl.umap` calls without `basis=` work on the dataset's preferred view."

In [ ]:
adata, info = p.load_dataset()\nadata.obsm[\"X_umap\"] = adata.obsm[\"X_umap_scANVI_dataset\"]  # default basis for sc.pl.umap\nprint(adata)\nprint(\"layers:\", list(adata.layers))\nprint(\"obsm:  \", list(adata.obsm))\nprint(\"sample_key   :\", info.sample_key)\nprint(\"cell_type_key:\", info.cell_type_key)\nprint(\"n_samples    :\", adata.obs[info.sample_key].nunique())

In [ ]:
sc.pl.umap(\n    adata,\n    color=[\"dataset\", \"Site\", \"Status\", \"severity_ord\", \"cell_coarse_aligned\"],\n    frameon=False,\n    ncols=2,\n)

## 2. Simple preprocessing on raw counts\n\nWe normalise the raw counts to 10k UMIs per cell and apply `log1p`. The log-normalised matrix becomes `adata.X` (also kept under `adata.layers[\"X_log1p\"]`); the raw counts move to `adata.layers[\"counts\"]` so PaSCient and other count-based methods can still pick them up by name."

In [ ]:
adata = p.preprocess(adata)\nprint(\"X is now log-normalised:\")\nprint(\"  X.max =\", float(adata.X.max()), \" X.min =\", float(adata.X.min()))\nprint(\"layers:\", list(adata.layers))

## 3. Sample-level metadata and train / val / test split\n\n`patpy.pp.extract_metadata` collapses cell-level `.obs` into one row per donor, keeping the columns that `DatasetInfo` advertises as sample-level. We add a 50 / 25 / 25 random split used downstream by SampleCLR's batch-aware sampler."

In [ ]:
metadata = p.build_metadata(adata, info)\nprint(metadata[\"split\"].value_counts())\nmetadata.head()

## 4. Build `meta_adata`\n\n`meta_adata.X` is set to the **mean pseudobulk of the log-normalised counts** — the simplest possible patient-level baseline. The same matrix is also stored in `obsm[\"X_pseudobulk_lognorm\"]` for symmetry with the other representations. The scANVI-pseudobulk (mean of the dataset-shipped integration embedding) and the CLR-transformed cell-type composition vector go straight into `obsm`."

In [ ]:
def maybe_compute(stage, compute_fn):\n    \"\"\"Load `obsm_<stage>.npz` if it exists, otherwise compute via `compute_fn`.\n\n    `compute_fn` must return a DataFrame indexed by sample id. The result is\n    persisted to `DATA_DIR / obsm_<stage>.npz` for cheap re-runs.\n    \"\"\"\n    obsm_path = p.DATA_DIR / f\"obsm_{stage}.npz\"\n    if obsm_path.exists():\n        print(f\"  [cache] {stage}: {obsm_path.name}\")\n        return p.load_obsm(stage)\n    df = compute_fn()\n    p.save_obsm(stage, df)\n    return df\n\n\npb_lognorm = maybe_compute(\n    \"pseudobulk_lognorm\",\n    lambda: p.pseudobulk_mean(adata, info, layer=p.LOGNORM_LAYER),\n)\npb_scanvi = maybe_compute(\n    \"pseudobulk_scanvi\",\n    lambda: p.pseudobulk_mean(adata, info, layer=p.SCANVI_OBSM),\n)\ncomp_clr = maybe_compute(\"composition_clr\", lambda: p.composition_clr(adata, info))\nprint(\"shapes:\", pb_lognorm.shape, pb_scanvi.shape, comp_clr.shape)

In [ ]:
sample_order = list(metadata.index)\npb_lognorm = pb_lognorm.reindex(sample_order)\n\nmeta_adata = ad.AnnData(\n    X=pb_lognorm.values.astype(np.float32),\n    obs=metadata,\n    var=pd.DataFrame(index=adata.var_names),\n)\nmeta_adata.layers[\"pseudobulk_lognorm\"] = meta_adata.X\nmeta_adata.uns[\"sample_key\"] = info.sample_key\nmeta_adata.uns[\"cell_type_key\"] = info.cell_type_key\n\nfor stage, df in [\n    (\"pseudobulk_lognorm\", pb_lognorm),\n    (\"pseudobulk_scanvi\",  pb_scanvi),\n    (\"composition_clr\",    comp_clr),\n]:\n    p.attach_obsm(meta_adata, stage, df)\n\nmeta_adata

## 5. SampleCLR self-supervised + fine-tuned\n\n`sampleclr.ContrastiveModel` is trained on the scANVI embedding (single batch-corrected latent space across the two cohorts). We first **pretrain** (self-supervised contrastive objective with the batch-aware sampler keyed on `Site`) and snapshot the embeddings, then **fine-tune** the same model with the `severity_idx` classification head and snapshot again. Both stages are heavy on GPU memory; the corresponding sbatch script lives at `scripts/stephenson_combat/02_sampleclr.sbatch`."

In [ ]:
# Train SampleCLR inline only if neither cached file exists. When training\n# happens here, pipeline.sampleclr_train ALSO persists the best ssl/ft state\n# dicts to MODELS_DIR/sampleclr_{ssl,ft}_state.pt for reproducibility.\nssl_path = p.DATA_DIR / \"obsm_sampleclr_ssl.npz\"\nft_path  = p.DATA_DIR / \"obsm_sampleclr_ft.npz\"\n\nif ssl_path.exists() and ft_path.exists():\n    print(\"[cache] using saved sampleclr embeddings\")\n    sclr_ssl = p.load_obsm(\"sampleclr_ssl\")\n    sclr_ft  = p.load_obsm(\"sampleclr_ft\")\nelse:\n    import torch\n    device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    sclr_ssl, sclr_ft, stats = p.sampleclr_train(\n        adata, info, metadata=metadata, device=device, save_checkpoints=True,\n    )\n    p.save_obsm(\"sampleclr_ssl\", sclr_ssl)\n    p.save_obsm(\"sampleclr_ft\",  sclr_ft)\n    print(stats)\n\nfor stage, df in [(\"sampleclr_ssl\", sclr_ssl), (\"sampleclr_ft\", sclr_ft)]:\n    p.attach_obsm(meta_adata, stage, df)\nprint(\"sampleclr_ssl:\", sclr_ssl.shape, \"sampleclr_ft:\", sclr_ft.shape)\nprint(\"checkpoints:\", p.SAMPLECLR_SSL_CKPT.exists(), p.SAMPLECLR_FT_CKPT.exists())

## 6. GloScope_py (GPU when available)\n\n`patpy.tl.GloScope_py` computes a symmetric KL distance between donors directly from a continuous embedding. With `use_gpu=True` it uses RAPIDS `cuML.NearestNeighbors`; the fallback is `pynndescent` on CPU. We feed it the scANVI embedding so the resulting distances are comparable across the two cohorts. The MDS embedding of the distance matrix becomes the `obsm` we store."

In [ ]:
def _gloscope_compute():\n    try:\n        import cuml  # noqa: F401\n        import cupy  # noqa: F401\n        import torch\n        use_gpu = torch.cuda.is_available()\n    except Exception as e:\n        print(f\"  cuML/cupy unavailable ({type(e).__name__}); falling back to CPU\")\n        use_gpu = False\n    df, distances = p.gloscope_py(adata, info, use_gpu=use_gpu)\n    np.savez(p.DATA_DIR / \"gloscope_py_distances.npz\",\n             distances=distances, samples=np.asarray(df.index))\n    return df\n\ngloscope_df = maybe_compute(\"gloscope_py\", _gloscope_compute)\np.attach_obsm(meta_adata, \"gloscope_py\", gloscope_df)\nprint(\"gloscope_py:\", gloscope_df.shape)

## 7. PaSCient (supervised patient encoder)\n\nPaSCient is trained from scratch on log-normalised counts with `severity_idx` as the donor-level classification target. We pass `normalize=False` because we already log-normalised `adata.X`; the hierarchical encoder yields a fixed-size patient embedding."

In [ ]:
def _pascient_compute():\n    import torch\n    device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    return p.pascient_train(adata, info, device=device, save_checkpoints=True)\n\npascient_df = maybe_compute(\"pascient\", _pascient_compute)\np.attach_obsm(meta_adata, \"pascient\", pascient_df)\nprint(\"pascient:\", pascient_df.shape)\nprint(\"checkpoint:\", p.PASCIENT_CKPT.exists())

## 8. Final `meta_adata` and quick visual sanity-check\n\n`meta_adata` now carries all seven representations. We persist it to `data/stephenson_combat/meta_adata.h5ad`, then take UMAPs over the supervised SampleCLR fine-tuned and the unsupervised GloScope representations as a sanity-check that severity gradients fall out where we'd expect."

In [ ]:
# Categorical order + colour maps for plotting (matches the original notebook).\nif \"Status\" in meta_adata.obs and hasattr(meta_adata.obs[\"Status\"], \"cat\"):\n    meta_adata.obs[\"Status\"] = meta_adata.obs[\"Status\"].cat.reorder_categories(\n        [\"Healthy\", \"Covid\", \"Flu\", \"Sepsis\"], ordered=False,\n    )\n    meta_adata.uns[\"Status_colors\"] = [\"lightblue\", \"salmon\", \"orange\", \"darkred\"]\n\nif \"severity_ord\" in meta_adata.obs and hasattr(meta_adata.obs[\"severity_ord\"], \"cat\"):\n    meta_adata.obs[\"severity_ord\"] = meta_adata.obs[\"severity_ord\"].cat.reorder_categories(\n        [\"nan\", \"healthy\", \"mild\", \"moderate\", \"severe\", \"critical\"], ordered=True,\n    )\n    meta_adata.uns[\"severity_ord_colors\"] = [\n        \"lightgray\", \"lightblue\", \"royalblue\", \"orange\", \"salmon\", \"darkred\",\n    ]\nmeta_adata

In [ ]:
# patpy benchmarking wants `meta_adata.obsm[<rep>]` (the embedding),\n# `obsm[<rep>_distances]` (pairwise distances) and `<rep>_neighbors` neighbor\n# graphs. `setup_benchmarking` populates all three plus per-rep UMAPs into\n# `obsm[\"X_umap_<rep>\"]`. For GloScope we feed in the original symmetric-KL\n# distance matrix saved by stage 04 instead of recomputing Euclidean.\ngloscope_dists_path = p.DATA_DIR / \"gloscope_py_distances.npz\"\nrepresentations = p.setup_benchmarking(\n    meta_adata,\n    gloscope_distances_path=gloscope_dists_path if gloscope_dists_path.exists() else None,\n)\nprint(\"representations:\", representations)

In [ ]:
PANELS = [\n    (\"pseudobulk_lognorm\", \"pseudobulk (log-norm)\"),\n    (\"pseudobulk_scanvi\",  \"pseudobulk (scANVI)\"),\n    (\"composition_clr\",    \"composition CLR\"),\n    (\"gloscope_py\",        \"GloScope_py\"),\n    (\"sampleclr_ssl\",      \"SampleCLR (SSL)\"),\n    (\"sampleclr_ft\",       \"SampleCLR (fine-tuned)\"),\n    (\"pascient\",           \"PaSCient\"),\n]\n\nfig, axes = plt.subplots(len(PANELS), 2, figsize=(10, 3.2 * len(PANELS)), squeeze=False)\nfor row, (rep, title) in enumerate(PANELS):\n    umap_key = f\"X_umap_{rep}\"\n    if umap_key not in meta_adata.obsm:\n        for ax in axes[row]:\n            ax.set_visible(False)\n        continue\n    meta_adata.obsm[\"X_umap\"] = meta_adata.obsm[umap_key]\n    sc.pl.umap(meta_adata, color=\"Status\",       ax=axes[row, 0], show=False, frameon=False, title=f\"{title}: Status\")\n    sc.pl.umap(meta_adata, color=\"severity_ord\", ax=axes[row, 1], show=False, frameon=False, title=f\"{title}: severity\")\nplt.tight_layout()\nplt.show()\nmeta_adata.obsm.pop(\"X_umap\", None)  # leave only the per-rep UMAPs on disk

## 9. Benchmark: KNN scores and trajectory preservation\n\nWe score each representation along three axes using `patpy.tl.evaluation.knn_prediction_score` (wrapped by `pipeline.benchmark_knn`):\n\n- **Relevant** (clinical information retention): KNN prediction of `severity_idx` (ranking, integer-coded severity), `Status` (classification), `outcome_bin` (classification).\n- **Technical** (batch mixing): KNN prediction of `Site` and `dataset`. Scores are inverted (`1 - score`) so higher = better batch mixing.\n- **Contextual**: KNN regression of `TimeSinceOnset`.\n\nA second benchmark uses `patpy.tl.evaluation.trajectory_correlation` to compute a diffusion pseudotime per representation (rooted on a Healthy donor) and Spearman-correlates it with `severity_idx`. Both tables are persisted alongside the final meta_adata.\n\nNote that the helper does an in-place `prepare_benchmark_columns` cast — `TimeSinceOnset`, `severity_idx`, `outcome_bin`, `Status_encoded` are all coerced to `float64` because sklearn KNeighborsRegressor cannot multiply strings or pandas `Int64`."

In [ ]:
knn_df = p.benchmark_knn(meta_adata)\nknn_df.to_csv(p.BENCHMARK_KNN_CSV, index=False)\nknn_pivot = knn_df.pivot_table(\n    index=\"representation\", columns=[\"covariate_type\", \"covariate\"], values=\"score\"\n).round(3)\nknn_pivot

In [ ]:
import seaborn as sns\n\nfig, ax = plt.subplots(figsize=(9, 5))\nsns.heatmap(knn_pivot, annot=True, cmap=\"RdYlGn\", center=0.5, ax=ax, cbar_kws={\"label\": \"KNN score (higher = better)\"})\nax.set_title(\"KNN prediction score per representation\\n(technical scores already inverted: higher = better batch mixing)\")\nplt.xticks(rotation=30, ha=\"right\")\nplt.tight_layout()\nplt.show()

In [ ]:
traj_df = p.benchmark_trajectory(meta_adata, trajectory_variable=\"severity_idx\")\ntraj_df.to_csv(p.BENCHMARK_TRAJECTORY_CSV)\ntraj_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))\ntraj_df.plot.barh(y=\"correlation\", ax=ax, legend=False, color=\"steelblue\")\nax.set_xlabel(\"Spearman corr. of diffusion pseudotime vs severity_idx\")\nax.set_title(\"Trajectory preservation per representation\")\nax.axvline(0, color=\"black\", lw=0.5)\nplt.tight_layout()\nplt.show()

## 10. Persist the final `meta_adata`\n\nSave everything (every `obsm[<rep>]`, every `obsm[<rep>_distances]`, every per-rep neighbor graph + UMAP, plus the benchmark tables) to a readably-named h5ad. Model checkpoints are already on disk under `data/.../models/`:\n\n- `models/sampleclr_ssl_state.pt`\n- `models/sampleclr_ft_state.pt`\n- `models/pascient_state.pt`\n\ntogether with `models/sampleclr_config.json` and `models/pascient_config.json` describing the hyperparameters used."

In [ ]:
meta_adata.write_h5ad(p.META_ADATA_FINAL)\nprint(\"final meta_adata:\", p.META_ADATA_FINAL)\nprint(\"  obsm keys :\", sorted(meta_adata.obsm.keys()))\nprint(\"  uns reps  :\", meta_adata.uns.get(\"sample_representations\"))\nprint(\"  benchmarks:\", p.BENCHMARK_KNN_CSV.name, p.BENCHMARK_TRAJECTORY_CSV.name)\nprint(\"  models    :\", *[c.name for c in [p.SAMPLECLR_SSL_CKPT, p.SAMPLECLR_FT_CKPT, p.PASCIENT_CKPT]])